In [ ]:
# Cell 1 — 공통 실행 환경 설정
# 이 Cell은 모든 tutorial Notebook의 첫 번째 Cell이며 여러 번 실행해도 안전합니다.

from pathlib import Path
import importlib.util
import os
import subprocess
import sys

# Google Drive의 project folder가 다르면 이 값만 수정합니다.
COLAB_PROJECT_ROOT = Path(
    "/content/drive/MyDrive/DiffDodree"
)

# 필요하면 환경변수로 project root를 직접 지정할 수 있습니다.
PROJECT_ROOT_OVERRIDE = os.environ.get("GUGAKGEN_PROJECT_ROOT")


def find_local_project_root(start: Path) -> Path:
    """현재 위치에서 pyproject.toml을 기준으로 project root를 찾습니다."""
    candidates = [start.resolve(), *start.resolve().parents]

    for candidate in candidates:
        if (candidate / "pyproject.toml").exists():
            return candidate

    raise FileNotFoundError(
        "project root를 찾지 못했습니다. "
        "GUGAKGEN_PROJECT_ROOT를 지정하거나 project folder에서 Notebook을 실행하세요."
    )


try:
    # 순수 local Python에는 상위 ``google`` namespace가 없을 수 있습니다.
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    # 이미 mount되어 있으면 기존 mount를 그대로 사용합니다.
    drive.mount("/content/drive", force_remount=False)

if PROJECT_ROOT_OVERRIDE:
    project_root = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
elif IN_COLAB:
    project_root = COLAB_PROJECT_ROOT.resolve()
else:
    project_root = find_local_project_root(Path.cwd())

if not project_root.exists():
    raise FileNotFoundError(
        f"project folder가 존재하지 않습니다: {project_root}\n"
        "COLAB_PROJECT_ROOT 또는 GUGAKGEN_PROJECT_ROOT를 확인하세요."
    )

bootstrap_script = project_root / "scripts" / "bootstrap_notebook.py"

if not bootstrap_script.exists():
    raise FileNotFoundError(
        f"Notebook bootstrap script를 찾지 못했습니다: {bootstrap_script}"
    )

environment_profile = "colab" if IN_COLAB else "local"

# 필요한 dependency만 확인하고 누락된 package만 설치합니다.
# Colab profile에서는 Colab이 제공한 CUDA용 PyTorch를 교체하지 않습니다.
subprocess.run(
    [
        sys.executable,
        str(bootstrap_script),
        "--project-root",
        str(project_root),
        "--profile",
        environment_profile,
        "--install-missing",
    ],
    check=True,
)

# 현재 kernel에서 src package를 즉시 import할 수 있게 합니다.
src_root = project_root / "src"

if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

os.environ["GUGAKGEN_PROJECT_ROOT"] = str(project_root)
os.chdir(project_root)

from gugakgen.device import resolve_device
from gugakgen.reproducibility import set_global_seed

runtime = resolve_device(preference="auto")
device = runtime.device

SEED = 2026
set_global_seed(SEED)

print("환경 설정 완료")
print(f"- 실행 환경: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print(f"- Project root: {project_root}")
print(f"- Device: {device}")
print(f"- AMP enabled: {runtime.amp_enabled}")
print(f"- Seed: {SEED}")

# 04. Model Sampling tutorial

이 Notebook은 공동연구자가 가장 자주 사용하는 interface입니다. pretrained Checkpoint를
불러온 뒤 Temperature, Graph guidance, Seed, transition path projection, 환입 mode를 조정하여
악보를 생성합니다. 모든 마디는 Exact-meter DP를 통과하므로 장단 합은 항상 정확합니다.

화면에서는 raw family path와 projected family path를 구분하여 표시합니다. MusicXML과 MIDI
export는 버튼 또는 마지막 fallback Cell에서 실행할 수 있습니다.

In [ ]:
from dataclasses import replace

import pandas as pd
from IPython.display import FileLink, clear_output, display

from gugakgen.config import load_yaml
from gugakgen.data.musicxml import tokens_to_score, write_musicxml
from gugakgen.workflows.generate import load_generation_bundle

checkpoint_candidates = [
    project_root / "artifacts/checkpoints/yangcheongdodeuri_maskgit.pt",
    project_root / "artifacts/m2_denoiser_v0.pt",
    project_root / "artifacts/checkpoints/yangcheongdodeuri_maskgit_smoke.pt",
]
checkpoint_path = next((path for path in checkpoint_candidates if path.exists()), None)
if checkpoint_path is None:
    raise FileNotFoundError("사용 가능한 Checkpoint가 없습니다. 먼저 03 Notebook을 실행하세요.")

bundle = load_generation_bundle(
    checkpoint_path,
    project_root=project_root,
    runtime=runtime,
)
piece_config = load_yaml(project_root / "configs/yangcheongdodeuri.yaml")
yulmyung = {int(midi): name for midi, name in piece_config["pitch"]["midi_to_yulmyung"].items()}

print("Checkpoint:", checkpoint_path.relative_to(project_root))
print("Checkpoint SHA-256:", bundle.checkpoint.sha256)
print("Device:", bundle.runtime.as_dict())
if "smoke" in checkpoint_path.name:
    print("주의: smoke Checkpoint는 interface 확인용이며 연구 결과용이 아닙니다.")

In [ ]:
# ipywidgets를 사용할 수 없는 환경에서는 DEFAULT_PARAMETERS를 그대로 사용합니다.
DEFAULT_PARAMETERS = {
    "seed": 2026,
    "num_steps": 12,
    "temperature_surface": 1.4,
    "temperature_family": 1.0,
    "graph_guidance": 0.5,
    "transition_projection": True,
    "exact_meter": True,
    "hwanip_mode": "learned",
}

try:
    import ipywidgets as widgets

    WIDGETS_AVAILABLE = True
    controls = {
        "seed": widgets.IntText(value=DEFAULT_PARAMETERS["seed"], description="Seed"),
        "num_steps": widgets.IntSlider(value=12, min=1, max=24, step=1, description="Sampling step"),
        "temperature_surface": widgets.FloatSlider(
            value=1.4, min=0.0, max=2.5, step=0.1, description="Surface Temperature"
        ),
        "temperature_family": widgets.FloatSlider(
            value=1.0, min=0.0, max=2.0, step=0.1, description="Family Temperature"
        ),
        "graph_guidance": widgets.FloatSlider(
            value=0.5, min=0.0, max=2.0, step=0.1, description="Graph guidance"
        ),
        "transition_projection": widgets.Checkbox(value=True, description="Path projection"),
        "exact_meter": widgets.Checkbox(value=True, disabled=True, description="Exact-meter DP"),
        "hwanip_mode": widgets.Dropdown(
            options=[("Learned relation", "learned"), ("Guided relation", "guided"), ("환입 끄기", "off")],
            value="learned",
            description="환입 mode",
        ),
    }
    display(widgets.VBox(list(controls.values())))
except ImportError:
    WIDGETS_AVAILABLE = False
    controls = {}
    print("ipywidgets를 사용할 수 없어 Python parameter fallback을 사용합니다.")


def current_parameters() -> dict:
    if not WIDGETS_AVAILABLE:
        return dict(DEFAULT_PARAMETERS)
    return {name: widget.value for name, widget in controls.items()}

In [ ]:
# 현재 control 값으로 한 개 악보를 생성합니다.
def generate_score(parameters: dict):
    if not parameters["exact_meter"]:
        raise ValueError("Phase 1에서는 Exact-meter DP를 해제할 수 없습니다.")

    sampling_config = replace(
        bundle.sampling_config,
        num_steps=int(parameters["num_steps"]),
        temperature_pitch_duration=float(parameters["temperature_surface"]),
        temperature_family=float(parameters["temperature_family"]),
        graph_guidance_scale=float(parameters["graph_guidance"]),
        hwanip_guidance_scale=1.0 if parameters["hwanip_mode"] == "guided" else 0.0,
        project_transition_path=bool(parameters["transition_projection"]),
    )

    original_relations = bundle.sampler.hwanip_relations
    if parameters["hwanip_mode"] == "off":
        bundle.sampler.hwanip_relations = None
    try:
        sampled = bundle.sampler.sample(
            seed=int(parameters["seed"]),
            config=sampling_config,
            score_anchors=bundle.score_anchors,
            family_anchors=bundle.family_anchors,
        )
    finally:
        bundle.sampler.hwanip_relations = original_relations

    score = tokens_to_score(
        sampled.tokens,
        bundle.context.vocabulary,
        piece_id=bundle.context.score.piece_id,
        part=bundle.context.score.part,
        meter_id_to_value=bundle.context.meter_id_to_value,
    )
    return sampled, score


def yul_string(measure) -> str:
    return " ".join(
        "쉼" if event.pitch is None else yulmyung.get(event.pitch, str(event.pitch))
        for event in measure.events
    )


def display_generation(sampled, score) -> None:
    raw_names = [bundle.context.vocabulary.decode_family(int(value)) for value in sampled.raw_family]
    final_names = [bundle.context.vocabulary.decode_family(int(value)) for value in sampled.tokens.family]
    rows = []
    for index, measure in enumerate(score.measures):
        rows.append(
            {
                "마디": index + 1,
                "장": measure.chapter,
                "raw family": raw_names[index],
                "projected family": final_names[index],
                "변경": raw_names[index] != final_names[index],
                "율명": yul_string(measure),
                "duration_q": [float(event.duration_q) for event in measure.events],
            }
        )
    table = pd.DataFrame(rows)
    print("Structure edit 수:", sampled.structure_edits)
    print("Final transition validity:", bundle.context.graph.transition_validity(sampled.tokens.family))
    display(table.head(16))
    display(table.loc[table["마디"].isin([1, 2, 3, 4, 25, 26, 27, 28])])


last_generation = {}
parameters = current_parameters()
sampled, generated_score = generate_score(parameters)
last_generation.update(parameters=parameters, sampled=sampled, score=generated_score)
display_generation(sampled, generated_score)

In [ ]:
# Button으로 parameter를 다시 읽고 생성 결과를 갱신합니다.
if WIDGETS_AVAILABLE:
    generate_button = widgets.Button(description="악보 생성", button_style="primary", icon="play")
    generation_output = widgets.Output()

    def on_generate_clicked(_button) -> None:
        with generation_output:
            clear_output(wait=True)
            parameters = current_parameters()
            sampled, generated_score = generate_score(parameters)
            last_generation.clear()
            last_generation.update(parameters=parameters, sampled=sampled, score=generated_score)
            display_generation(sampled, generated_score)

    generate_button.on_click(on_generate_clicked)
    display(generate_button, generation_output)

In [ ]:
# MusicXML/MIDI export 함수와 Button을 준비합니다.
def export_last_generation() -> tuple[Path, Path]:
    if not last_generation:
        raise RuntimeError("먼저 악보를 생성하세요.")
    output_dir = project_root / "artifacts/tutorial_outputs"
    output_dir.mkdir(parents=True, exist_ok=True)
    seed = int(last_generation["parameters"]["seed"])
    xml_path = output_dir / f"yangcheongdodeuri_seed_{seed}.musicxml"
    midi_path = output_dir / f"yangcheongdodeuri_seed_{seed}.mid"
    export_config = piece_config["export"]
    write_musicxml(
        last_generation["score"],
        xml_path,
        divisions_per_quarter=export_config["divisions_per_quarter"],
        key_fifths=export_config["key_fifths"],
        part_id=export_config["part_id"],
        part_name=export_config["part_name"],
    )

    from music21 import converter

    converter.parse(str(xml_path)).write("midi", fp=str(midi_path))
    print("MusicXML:", xml_path.relative_to(project_root))
    print("MIDI:", midi_path.relative_to(project_root))
    display(FileLink(str(xml_path)), FileLink(str(midi_path)))
    return xml_path, midi_path


if WIDGETS_AVAILABLE:
    export_button = widgets.Button(description="MusicXML/MIDI 내보내기", icon="download")
    export_output = widgets.Output()

    def on_export_clicked(_button) -> None:
        with export_output:
            clear_output(wait=True)
            export_last_generation()

    export_button.on_click(on_export_clicked)
    display(export_button, export_output)

In [ ]:
# Restart and Run All에서 자동 export가 필요하면 True로 바꿉니다.
EXPORT_ON_RUN_ALL = False
if EXPORT_ON_RUN_ALL:
    export_last_generation()

## 결과 해석

`raw family`는 Transformer와 guidance가 만든 직접 출력이고, `projected family`는 관찰된
transition edge만 통과하도록 Viterbi projection을 적용한 결과입니다. 따라서 논문에서는
두 결과를 분리해 보고해야 합니다. MIDI는 악보 확인을 위한 편의 출력이며 피리 음색,
시김새, 농현 또는 실제 연주 표현을 재현하지 않습니다.